# Continuazione OpenMM: da 1 ns a 10 ns

Questo notebook **prosegue** le simulazioni di **F0010, CF06 e CF02** dallo stato finale a 1 ns per altri 9 ns. Non crea un nuovo sistema e non ripete minimizzazione, NVT o NPT.

Per ciascun candidato usa il `system.xml` originale, la topologia solvata e lo `State` finale (posizioni, velocità e box periodico). Gli output della prosecuzione sono scritti in una directory separata.

In [ ]:
from pathlib import Path
import math

try:
    import openmm
    from openmm import XmlSerializer, Platform, unit
    from openmm.app import PDBFile, Simulation, DCDReporter, StateDataReporter
except ImportError:  # compatibilità con installazioni OpenMM precedenti
    import simtk.openmm as openmm
    from simtk.openmm import XmlSerializer, Platform
    from simtk import unit
    from simtk.openmm.app import PDBFile, Simulation, DCDReporter, StateDataReporter

print(f"OpenMM {getattr(openmm, '__version__', 'versione non disponibile')}")

## Configurazione centrale

Modificare soltanto `BASE_DIR` se la cartella `production_1ns` si trova altrove. `CPU` è la scelta predefinita, portabile e robusta; si può impostare `CUDA`, `OpenCL` o un'altra piattaforma disponibile.

In [ ]:
BASE_DIR = Path("/percorso/alla/cartella/production_1ns").expanduser()
candidates = ["F0010", "CF06", "CF02"]
continuation_ns = 9.0
report_interval_ps = 2.0
platform_name = "CPU"  # alternative comuni: CUDA, OpenCL, Reference
platform_properties = {}  # esempio CPU: {"Threads": "8"}

temperature = 300.0 * unit.kelvin
friction = 1.0 / unit.picosecond
timestep = 2.0 * unit.femtoseconds

steps = int(round(continuation_ns * 1000.0 / timestep.value_in_unit(unit.picoseconds)))
report_interval_steps = int(round(report_interval_ps / timestep.value_in_unit(unit.picoseconds)))
assert steps > 0 and report_interval_steps > 0
print(f"Passi per candidato: {steps:,}; intervallo reporter: {report_interval_steps:,} passi")

## Controlli preliminari

I controlli seguenti si fermano prima del calcolo se manca un file, se la piattaforma non è disponibile o se il numero di atomi della PDB non coincide con il numero di particelle del sistema serializzato.

In [ ]:
def input_paths(candidate):
    root = BASE_DIR / candidate
    return {
        "system": root / "states" / f"{candidate}_system.xml",
        "pdb": root / "structures" / f"{candidate}_BD_solvated.pdb",
        "state": root / "states" / f"{candidate}_production_1ns_final_state.xml",
    }

if not BASE_DIR.is_dir():
    raise FileNotFoundError(f"BASE_DIR non esiste o non è una directory: {BASE_DIR}")

available_platforms = [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
if platform_name not in available_platforms:
    raise ValueError(f"Piattaforma {platform_name!r} non disponibile. Disponibili: {available_platforms}")

validated = {}
for candidate in candidates:
    paths = input_paths(candidate)
    missing = [str(path) for path in paths.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"File mancanti per {candidate}:\n  " + "\n  ".join(missing))

    with paths["system"].open() as handle:
        system = XmlSerializer.deserialize(handle.read())
    pdb = PDBFile(str(paths["pdb"]))
    n_system = system.getNumParticles()
    n_pdb = pdb.topology.getNumAtoms()
    if n_system != n_pdb:
        raise ValueError(f"{candidate}: system.xml ha {n_system} particelle, PDB ha {n_pdb} atomi")

    force_names = [system.getForce(i).__class__.__name__ for i in range(system.getNumForces())]
    barostats = [name for name in force_names if "Barostat" in name]
    if not barostats:
        raise ValueError(f"{candidate}: nessun barostato trovato nel system.xml; il notebook non lo aggiunge automaticamente")

    validated[candidate] = {"paths": paths, "atoms": n_system, "barostats": barostats}
    print(f"OK {candidate}: {n_system:,} atomi; barostato: {', '.join(barostats)}")

## Prosecuzione delle tre simulazioni

Ogni candidato viene elaborato in sequenza per contenere l'uso di memoria. `Context.setState(...)` ripristina posizioni, velocità, vettori del box periodico e parametri di contesto salvati nello stato finale a 1 ns. Il `MonteCarloBarostat` viene riutilizzato dal `system.xml`; non ne viene aggiunto un secondo.

In [ ]:
def continue_candidate(candidate):
    paths = validated[candidate]["paths"]
    output_root = BASE_DIR / candidate / "continuation_1ns_to_10ns"
    trajectory_dir = output_root / "trajectory"
    logs_dir = output_root / "logs"
    states_dir = output_root / "states"
    for directory in (trajectory_dir, logs_dir, states_dir):
        directory.mkdir(parents=True, exist_ok=True)

    dcd_path = trajectory_dir / f"{candidate}_continuation_1ns_to_10ns.dcd"
    log_path = logs_dir / f"{candidate}_continuation_1ns_to_10ns.csv"
    final_state_path = states_dir / f"{candidate}_production_10ns_final_state.xml"
    existing = [path for path in (dcd_path, log_path, final_state_path) if path.exists()]
    if existing:
        raise FileExistsError("Output già presenti; spostarli o rinominarli prima di rilanciare:\n  " + "\n  ".join(map(str, existing)))

    with paths["system"].open() as handle:
        system = XmlSerializer.deserialize(handle.read())
    with paths["state"].open() as handle:
        saved_state = XmlSerializer.deserialize(handle.read())
    pdb = PDBFile(str(paths["pdb"]))

    integrator = openmm.LangevinMiddleIntegrator(temperature, friction, timestep)
    platform = Platform.getPlatformByName(platform_name)
    simulation = Simulation(pdb.topology, system, integrator, platform, platform_properties)
    simulation.context.setState(saved_state)

    restored = simulation.context.getState(getPositions=True, getVelocities=True)
    if len(restored.getPositions()) != system.getNumParticles():
        raise RuntimeError(f"{candidate}: lo stato ripristinato ha un numero inatteso di posizioni")
    if len(restored.getVelocities()) != system.getNumParticles():
        raise RuntimeError(f"{candidate}: lo stato ripristinato ha un numero inatteso di velocità")

    simulation.reporters.append(DCDReporter(str(dcd_path), report_interval_steps, enforcePeriodicBox=True))
    simulation.reporters.append(StateDataReporter(
        str(log_path), report_interval_steps, step=True, time=True,
        potentialEnergy=True, kineticEnergy=True, totalEnergy=True,
        temperature=True, volume=True, density=True, speed=True,
        progress=True, remainingTime=True, totalSteps=steps, separator=','
    ))

    print(f"\n{candidate}: prosecuzione di {continuation_ns:g} ns su {platform_name}...")
    simulation.step(steps)
    final_state = simulation.context.getState(
        getPositions=True, getVelocities=True, getParameters=True,
        getEnergy=True, enforcePeriodicBox=True
    )
    final_state_path.write_text(XmlSerializer.serialize(final_state))
    print(f"Completato {candidate}: {output_root}")
    del simulation, integrator, saved_state, final_state
    return output_root

outputs = {candidate: continue_candidate(candidate) for candidate in candidates}
outputs

## Istruzioni finali per il prof. Leoni

1. Installare una versione di OpenMM compatibile con quella usata per serializzare i file XML.
2. Copiare la cartella `production_1ns` mantenendo le sottocartelle `states/` e `structures/` di ciascun candidato.
3. Impostare `BASE_DIR` nella cella di configurazione. Lasciare `platform_name = "CPU"` per la massima portabilità, oppure scegliere una piattaforma disponibile.
4. Eseguire le celle dall'alto verso il basso. Il calcolo parte direttamente dallo stato finale a 1 ns e aggiunge 9 ns.
5. Per ogni candidato, recuperare DCD, CSV e stato XML finale da `continuation_1ns_to_10ns/`. Il DCD contiene solo i 9 ns aggiuntivi; il tempo riportato nel log continua dal tempo memorizzato nello `State`, se presente.

### Parametri della simulazione originale

- Force field: AMBER ff14SB
- Acqua: TIP3P
- Temperatura: 300 K
- Pressione: 1 bar
- Ionic strength: 0.15 M
- Padding solvente: 1.0 nm
- PME cutoff: 1.0 nm
- Constraints: HBonds
- Integratore: LangevinMiddleIntegrator; friction 1 ps⁻¹; timestep 2 fs
- Equilibrazione originale: NVT 100 ps + NPT 100 ps
- Produzione iniziale: 1 ns
- Frame interval: 2 ps
- Random seed base originale: 20260808

Il force field, il solvente, i vincoli, PME e il barostato sono già incorporati nel `system.xml`. Questo notebook non ricostruisce né modifica il sistema.